# Does the tin need finding, or just centring?

**The question this notebook exists to answer.** Herbatka can ask people to put the tin in
the middle of the frame. Once it does, a fixed centre crop is free, instant, and has no
model to train, no labels to draw and no licence to worry about. A learned segmenter costs
all four.

So the segmenter is not the starting point — it is the thing that has to *win*. This
notebook builds the two free baselines, measures them, and states in advance what result
would justify training anything at all.

Three strategies, cheapest first:

| Strategy | Learned params | Dependencies | What it can't do |
|---|---|---|---|
| `full_frame` | 0 | none | everything; this is the floor |
| `center_crop` | 0 | none | adapt to how much of the frame the tin fills |
| `grabcut_crop` | 0 | opencv | survive a busy background or a tin that isn't centred |

Two numbers decide it: **tokens** (what a crop saves on every single call, forever) and
**read accuracy** (what a crop costs if it clips the label). A crop that halves the bill
and drops one word of the brand name is a bad crop.

In [2]:
import matplotlib.pyplot as plt
import numpy as np

from herbatka_vision import crops, paths

# Anchored on the package, not on the working directory, so this behaves identically in
# VS Code, JupyterLab and a plain shell. If the import fails, the kernel is not the one in
# vision/.venv — see the README.
paths.ensure()
RAW, OUT = paths.RAW, paths.OUTPUTS

plt.rcParams["figure.dpi"] = 110
print("photo directory:", RAW)

photo directory: /Users/sasha/pet-projects/herbatka/vision/data/raw


## 1. The photographs

Drop your tin photos into `vision/data/raw/`. That directory is gitignored — the pictures
are large, binary, and pictures of your kitchen.

**Shoot for the failure modes, not for the average.** Twenty easy photographs will tell you
everything is fine. What you actually want, and roughly in these proportions:

- tins filling ~20% of the frame and tins filling ~80% (this is the variance a fixed crop
  cannot absorb, and the whole case for GrabCut)
- glare: a window, a ceiling light, anything that puts a bright band across metal
- a busy background — a full shelf, a patterned counter
- a hand holding the tin
- at least a few tins whose label is *not* in Latin script, if you own any

Thirty photographs is enough to run this notebook honestly. A hundred is enough to trust it.

In [ ]:
photos = crops.load_images(RAW)

if not photos:
    print(f"No photographs in {RAW} yet.")
    print("Drop 30+ tin photos there and re-run from here. See the guidance above.")
else:
    sizes = np.array([img.shape[:2] for _, img in photos])
    tokens = np.array([crops.image_tokens(w, h) for h, w in sizes])
    print(f"{len(photos)} photographs")
    print(f"  median size   {np.median(sizes[:, 1]):.0f} x {np.median(sizes[:, 0]):.0f}")
    print(f"  median tokens {np.median(tokens):.0f}  (full frame, uncropped)")

## 2. What the three strategies actually do

A contact sheet is worth more than an IoU here, because at this stage you are not looking
for a number — you are looking for the specific ways GrabCut fails, so you know whether the
failures are the kind a learned model would fix.

Watch for two things in particular. **Glare splitting the tin** into two blobs (handled in
`crops.largest_component`, but check that the surviving blob is the right one). And
**GrabCut swallowing the countertop** when the tin's colour is close to the surface it
stands on — that one is not fixable classically, and it is the strongest argument for
training a segmenter.

In [ ]:
def contact_sheet(photos, n=4, seed=0):
    if not photos:
        print("No photographs loaded — see section 1.")
        return
    rng = np.random.default_rng(seed)
    picks = rng.choice(len(photos), size=min(n, len(photos)), replace=False)

    names = list(crops.STRATEGIES)
    fig, axes = plt.subplots(len(picks), len(names), figsize=(3.2 * len(names), 3.4 * len(picks)))
    axes = np.atleast_2d(axes)

    for row, i in enumerate(picks):
        path, img = photos[i]
        for col, name in enumerate(names):
            crop = crops.STRATEGIES[name](img)
            ax = axes[row, col]
            ax.imshow(crop.image)
            ax.set_xticks([])
            ax.set_yticks([])
            if row == 0:
                ax.set_title(name, fontsize=10)
            ax.set_xlabel(f"{crop.tokens} tok", fontsize=8)
        axes[row, 0].set_ylabel(path.name[:18], fontsize=7)

    fig.tight_layout()
    return fig


contact_sheet(photos)

## 3. What cropping saves

Tokens scale with pixel count, so this is the one number that is knowable before you have
any accuracy data at all — and it is knowable *per photograph*, forever, on every call the
app ever makes.

Treat the saving as the budget the crop has to justify. If `center_crop` cuts 55% of the
tokens and `grabcut_crop` cuts 70%, GrabCut's extra 15 points is what it must buy without
losing any read accuracy. That is a small prize for a fragile algorithm — and noticing that
early is the point of measuring before building.

In [ ]:
def token_table(photos):
    if not photos:
        print("No photographs loaded — see section 1.")
        return None
    rows = {}
    for name, fn in crops.STRATEGIES.items():
        rows[name] = [fn(img).tokens for _, img in photos]

    base = np.mean(rows["full_frame"])
    print(f"{'strategy':<14} {'mean tok':>9} {'median':>8} {'vs full':>9}")
    print("-" * 44)
    for name, vals in rows.items():
        v = np.array(vals)
        print(f"{name:<14} {v.mean():>9.0f} {np.median(v):>8.0f} {1 - v.mean() / base:>8.0%}")
    return rows


token_rows = token_table(photos)

## 4. Sweeping the crop fraction

`center_crop(frac=0.65)` is a guess. The right value is a fact about how *your* users frame
a photograph, and it is one line to find out.

The curve you are looking for has a knee: tokens fall smoothly as `frac` shrinks, but read
accuracy falls off a cliff the moment the crop starts clipping labels. Section 5 supplies
the accuracy half — until then this only shows you the cheap half, which always looks like
"smaller is better". Do not pick a value from this cell alone.

In [ ]:
fracs = np.arange(0.35, 1.01, 0.05)

if photos:
    means = [np.mean([crops.center_crop(img, frac=f).tokens for _, img in photos]) for f in fracs]
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.plot(fracs, means, marker="o", ms=4)
    ax.set_xlabel("center_crop frac")
    ax.set_ylabel("mean image tokens")
    ax.set_title("Cheap half of the tradeoff — accuracy is section 5")
    ax.grid(alpha=0.3)
    fig.tight_layout()
else:
    print("No photographs loaded — see section 1.")

## 5. The half that actually decides it

Everything above is free to compute and proves nothing on its own. A crop of zero pixels
saves 100% of the tokens.

The measurement that settles the question is **how much of the tin's text survives the
crop**, and for that you need ground truth: for a sample of photographs, what the tin
actually says. Transcribe 20–30 by hand into `data/transcripts.json` as
`{"filename.jpg": "TWININGS\nEarl Grey\n..."}`. It is a dull hour and it is the single
highest-value hour in this project — it is the only thing here that can tell you a crop
made the reading *worse*.

OCR is optional (`uv sync --group ocr`). Without it this section reports what it needs and
stops, rather than pretending.

In [ ]:
import json

TRANSCRIPTS = paths.TRANSCRIPTS

try:
    from rapidocr_onnxruntime import RapidOCR

    _ocr = RapidOCR()

    def read_text(img):
        result, _ = _ocr(img)
        return "\n".join(line[1] for line in result) if result else ""

    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False

    def read_text(img):
        raise RuntimeError("OCR not installed: uv sync --group ocr")


print("OCR available:      ", OCR_AVAILABLE)
print("transcripts present:", TRANSCRIPTS.exists())

In [ ]:
def char_accuracy(truth: str, got: str) -> float:
    """1 - normalised edit distance, on casefolded alphanumerics only.

    Punctuation and whitespace are stripped because OCR disagreements about them are not
    errors anyone in this app cares about: the downstream matcher normalises them away
    before it ever compares two names.
    """

    def keep(s):
        return "".join(c for c in s.casefold() if c.isalnum())

    a, b = keep(truth), keep(got)
    if not a:
        return 1.0 if not b else 0.0

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return max(0.0, 1 - prev[-1] / len(a))


def accuracy_table(photos):
    if not (OCR_AVAILABLE and TRANSCRIPTS.exists() and photos):
        print("Needs all three: photographs, transcripts.json, and the ocr group installed.")
        return None

    truth = json.loads(TRANSCRIPTS.read_text())
    labelled = [(p, img) for p, img in photos if p.name in truth]
    print(f"scoring {len(labelled)} labelled photographs\n")

    print(f"{'strategy':<14} {'char acc':>9} {'mean tok':>9}")
    print("-" * 35)
    results = {}
    for name, fn in crops.STRATEGIES.items():
        accs, toks = [], []
        for path, img in labelled:
            crop = fn(img)
            accs.append(char_accuracy(truth[path.name], read_text(crop.image)))
            toks.append(crop.tokens)
        results[name] = (float(np.mean(accs)), float(np.mean(toks)))
        print(f"{name:<14} {np.mean(accs):>9.1%} {np.mean(toks):>9.0f}")
    return results


accuracy_results = accuracy_table(photos)

## 6. The decision, written down before the result

Committing to this now is what stops the next fortnight from being a segmenter you build
because you had already decided to build one.

**Train a segmenter only if** `grabcut_crop` loses more than ~2 points of character accuracy
against `center_crop`, *or* it fails visibly on more than ~15% of the contact sheet. Those
are the two failure shapes a learned model genuinely fixes.

**Ship `center_crop` and move on if** it lands within a point or two of GrabCut on accuracy
while saving most of the tokens. That is a real result, it is the likeliest one given the
centring instruction, and it frees the whole project to go at the part that is actually
hard — reading the label and resolving it to a `Tea` row.

**Two things stay worth building either way**, and neither is a segmenter:

1. **Cylindrical unwarping** needs a silhouette, and straightening curved label text is the
   biggest single lever on OCR accuracy available here. If GrabCut's mask is good enough to
   unwarp from, you get it without training anything.
2. **Background compositing** needs a mask, and pasting real tins onto random backgrounds is
   how the embedding model in the next phase gets its training volume.

So a "GrabCut is fine" result does not end the mask work — it just means the mask is worth
having for reasons other than cropping.

In [ ]:
if accuracy_results:
    ref = accuracy_results["center_crop"]
    for name, (acc, tok) in accuracy_results.items():
        if name == "center_crop":
            continue
        print(
            f"{name:>14}: {acc - ref[0]:+.1%} accuracy, "
            f"{1 - tok / ref[1]:+.0%} tokens vs center_crop"
        )
    print("\nRule: train a segmenter only if grabcut_crop is >2 points better on accuracy.")
else:
    print("Run section 5 first.")

## Next

1. Take the photographs — 30 minimum, weighted toward the failure modes in section 1.
2. Transcribe 20–30 of them into `data/transcripts.json`.
3. Run sections 2–5 and apply the rule in section 6.
4. Whatever the rule says, the next notebook is the reading step: OCR → `TinReading`
   (see `herbatka_vision/schema.py`) → matched against `Brand` and `Tea`.

`schema.py` is already pinned to the eight `tea_type` values and four `caffeine_level`
values that `api/app/models/catalog.py` will actually accept, so a reading that validates
is a reading the database can store.